# P3 — Trous, cardinalité, cibles

**Enchaînement : 1.6 Trous → 1.4 Cardinalité (attend 1.3 Typer) → 1.9 Cibles**

Phase 1 du projet *Santander Satisfaction* : connaître les données avant de modéliser.
Chaque tâche suit le même plan : **la question**, **la mesure** (le code), **le constat**, **la décision**
que le porteur doit pouvoir défendre seul.

Pour relancer : *Exécuter tout*. Les données sont lues dans `data/` (récupérées par `git pull`), tables entières.

In [1]:
# --- Réglages communs à toute l'équipe ------------------------------------
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

# Les données sont dans data/ à la racine du dépôt (récupérées par git pull)
DATA = next(p for p in (Path("data"), Path("../data")) if (p / "train.csv").is_file())
CIBLE, ID = "TARGET", "ID"
GRAINE = 42             # même graine pour toute l'équipe : même découpage
PART_VALIDATION = 0.20  # protocole : 80 % apprentissage / 20 % validation, stratifié
PLANCHER = 0.6890       # AUC de la règle métier, donnée par l'énoncé
SENTINELLES = (-999999, 9999999999)
FAMILLES = ("saldo", "imp", "num", "ind", "delta")

t0 = time.perf_counter()
train = pd.read_csv(DATA / "train.csv")
a_decider = pd.read_csv(DATA / "a_decider.csv")
DUREE_CHARGEMENT = time.perf_counter() - t0
VARIABLES = [c for c in train.columns if c not in (ID, CIBLE)]
y = train[CIBLE]


def famille(col):
    if col in (ID, CIBLE):
        return col
    debut = col.split("_")[0]
    return debut if debut in FAMILLES else "isolée"


def pct(x):
    return f"{100 * x:.2f} %"


print(f"train : {train.shape} · a_decider : {a_decider.shape} · lus en {DUREE_CHARGEMENT:.1f} s depuis {DATA}")

from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, train_test_split


def profil(groupes, nom):
    """Taux de mécontents et effectif par groupe."""
    t = groupes.agg(["mean", "size"])
    return pd.DataFrame({"taux de mécontents": [pct(v) for v in t["mean"]], "clients": t["size"].to_numpy()},
                        index=pd.Index(t.index, name=nom))

train : (60816, 371) · a_decider : (15204, 370) · lus en 0.9 s depuis ../data


## 1.6 — Trous

**Question** : où manque-t-il de l'information, et comment ce manque est-il codé ?

In [2]:
print("Valeurs vides (NaN) :", int(train.isna().sum().sum() + a_decider.isna().sum().sum()))

Valeurs vides (NaN) : 0


Aucun `NaN` : les trous existent pourtant, mais **codés**. On en trouve quatre.

### `var3 = -999999` — un vrai manquant

In [3]:
m = train["var3"] == -999999
print(f"{int(m.sum())} clients dans train, {int((a_decider['var3'] == -999999).sum())} dans a_decider")
print(f"Taux de mécontents : {pct(y[m].mean())} ({int(y[m].sum())} mécontent) contre {pct(y[~m].mean())}")
print(f"La valeur 2 occupe {pct((train['var3'] == 2).mean())} des clients")

88 clients dans train, 28 dans a_decider
Taux de mécontents : 1.14 % (1 mécontent) contre 3.96 %
La valeur 2 occupe 97.53 % des clients


**Décision : remplacer par `2`**, la valeur dominante. Un indicateur « manquant » reposerait sur **1 seul mécontent** : il n'apprendrait rien de fiable.

### `delta_ = 9999999999` — pas un manquant : une apparition

In [4]:
DELTA = [c for c in VARIABLES if famille(c) == "delta"]
s = train["delta_imp_aport_var13_1y3"] == 9999999999
print(f"delta_imp_aport_var13_1y3 : {int(s.sum())} lignes codées")
display(train.loc[s, ["imp_aport_var13_hace3", "imp_aport_var13_ult1"]].describe().loc[["min", "max"]])
moins1 = train["delta_imp_aport_var13_1y3"] == -1
print(f"Code -1 : {int(moins1.sum())} lignes")
display(train.loc[moins1, ["imp_aport_var13_hace3", "imp_aport_var13_ult1"]].describe().loc[["min", "max"]])
fini = pd.concat([train[DELTA], a_decider[DELTA]]).replace(9999999999, np.nan)
print(f"Plus grande variation finie, toutes colonnes delta_ : {np.nanmax(fini.to_numpy()):.2f}")
apparition = (train[DELTA] == 9999999999).any(axis=1)
print(f"Clients avec une apparition : {int(apparition.sum())}, taux {pct(y[apparition].mean())} contre {pct(y[~apparition].mean())}")

delta_imp_aport_var13_1y3 : 291 lignes codées


,imp_aport_var13_hace3,imp_aport_var13_ult1
min,0.0000,300.0000
max,0.0000,450000.0000


Code -1 : 1343 lignes


,imp_aport_var13_hace3,imp_aport_var13_ult1
min,300.0000,0.0000
max,840000.0000,0.0000


Plus grande variation finie, toutes colonnes delta_ : 8.00
Clients avec une apparition : 421, taux 1.90 % contre 3.97 %


Quand le code apparaît, le montant d'il y a 3 mois (`hace3`) vaut **toujours 0** et le montant récent (`ult1`) est **toujours positif** : la variation part de zéro, c'est une division par zéro codée « infini ». Symétriquement, `-1` = le montant a **disparu** (base > 0, récent = 0).

**Décision : remplacer `9999999999` par `9`**, juste au-dessus de la plus grande variation finie (8). L'ordre est gardé (une apparition reste la plus forte hausse), sans écraser l'échelle. Ce n'est **pas** un trou : aucune imputation.

### `var38 = 117310.979016…` — un trou déguisé

In [5]:
vc = train["var38"].value_counts()
m38 = train["var38"] == vc.index[0]
print(f"var38 : {train['var38'].nunique()} valeurs différentes")
print(f"La plus fréquente, {vc.index[0]}, revient {int(m38.sum())} fois ({pct(m38.mean())}) ; la suivante {int(vc.iloc[1])} fois")
print(f"Taux de mécontents : {pct(y[m38].mean())} contre {pct(y[~m38].mean())}")

var38 : 46577 valeurs différentes
La plus fréquente, 117310.979016494, revient 12012 fois (19.75 %) ; la suivante 12 fois
Taux de mécontents : 4.06 % contre 3.93 %


Un montant à six décimales partagé par un client sur cinq n'est pas un montant : c'est une **valeur de remplissage** (probablement une moyenne). Mais le taux de mécontents est le même dans les deux groupes.

**Décision : garder `var38` telle quelle.** **Essai à journaliser au bloc 2** : remplacer la valeur par un manquant + un indicateur `var38_rempli`, et mesurer si l'AUC bouge.

### `var36 = 99` — un code d'absence qui parle

In [6]:
profil(train.groupby("var36")[CIBLE], "var36")

,taux de mécontents,clients
var36,,
0,0.00 %,321
1,3.03 %,11718
2,2.75 %,6955
3,1.63 %,17784
99,6.54 %,24038


`99` concerne **40 %** des clients et porte le **plus fort** taux de mécontents. Lu comme un nombre, 99 serait « très loin » de 3 : c'est faux, c'est une **catégorie**.

**Décision : encoder `var36` en catégories** (une colonne 0/1 par valeur). Ne pas l'imputer : l'absence est elle-même une information.

### Les zéros — pas des trous

In [7]:
zeros = (train[VARIABLES] == 0).mean()
print(f"Colonnes nulles à plus de 99 % : {int((zeros > 0.99).sum())} sur {len(VARIABLES)}")
non_nuls = (train[VARIABLES] != 0).sum(axis=1)
t = profil(train.groupby(pd.qcut(non_nuls, 5, duplicates="drop"), observed=True)[CIBLE], "valeurs non nulles")
t.index = [f"{int(i.left) + 1} à {int(i.right)}" for i in t.index]
t

Colonnes nulles à plus de 99 % : 217 sur 369


,taux de mécontents,clients
8 à 14,8.43 %,12709
15 à 29,2.90 %,19415
30 à 32,2.07 %,5468
33 à 49,2.47 %,11383
50 à 149,3.18 %,11841


Un zéro dit « pas de produit, pas d'opération » : c'est une **mesure**, pas un manque. Et le **nombre de valeurs non nulles** sépare déjà les mécontents : 8,4 % chez les clients les moins équipés, 2 à 3 % ailleurs.

**Décision : garder les zéros** (les remplacer par une moyenne détruirait l'information). **Essai proposé au constructeur** : ajouter la variable « nombre de valeurs non nulles ».

## 1.4 — Cardinalité

**Préalable (1.3, P2)** : toutes les colonnes sont numériques et les `ind_` valent 0 ou 1. On peut compter les valeurs distinctes sans conversion.

**Question** : combien de valeurs différentes prend chaque variable ? Lesquelles ne servent à rien ?

### Colonnes constantes

In [8]:
n_tr, n_de = train[VARIABLES].nunique(), a_decider[VARIABLES].nunique()
CONSTANTES = sorted(n_tr[n_tr == 1].index)
print("Constantes sur les 4 000 premières lignes :", int((train[VARIABLES].head(4000).nunique() == 1).sum()))
print("Constantes sur tout train                 :", len(CONSTANTES))
print("Constantes dans train mais variables dans a_decider :", list(n_tr[(n_tr == 1) & (n_de > 1)].index))

Constantes sur les 4 000 premières lignes : 115
Constantes sur tout train                 : 36
Constantes dans train mais variables dans a_decider : ['imp_reemb_var17_hace3', 'num_reemb_var17_hace3']


L'énoncé annonce **97** constantes, mesurées sur un échantillon de 4 000 lignes. Sur la table entière, il n'y en a que **36** : un échantillon rend constantes des colonnes qui ne le sont pas. Deux d'entre elles varient dans `a_decider`, mais le modèle n'en a jamais vu qu'une valeur : il ne peut rien en apprendre.

**Décision : supprimer les 36 colonnes constantes sur train**, mesurées sur la table entière.

### Colonnes quasi constantes

In [9]:
lignes = []
for c in VARIABLES:
    if n_tr[c] == 1:
        continue
    rare = train[c] != train[c].value_counts().index[0]
    if rare.mean() <= 0.001:  # une valeur occupe au moins 99,9 % des lignes
        k = int(y[rare].sum())
        lignes.append((c, int(rare.sum()), k, pct(y[rare].mean()), binomtest(k, int(rare.sum()), y.mean()).pvalue))
quasi = pd.DataFrame(lignes, columns=["colonne", "clients rares", "mécontents parmi eux", "taux", "p"]).set_index("colonne")
SEUIL = 0.01 / len(quasi)  # correction de Bonferroni : on teste toutes les colonnes à la fois
print(f"{len(quasi)} quasi constantes ; mécontents parmi les clients rares :",
      quasi["mécontents parmi eux"].value_counts().sort_index().to_dict())
print(f"Seuil du test : p < {SEUIL:.0e}")
QUASI_GARDEES = list(quasi.index[quasi["p"] < SEUIL])
QUASI_SUPPRIMEES = list(quasi.index[quasi["p"] >= SEUIL])
quasi.loc[QUASI_GARDEES].assign(p=lambda d: d["p"].map("{:.1e}".format))

110 quasi constantes ; mécontents parmi les clients rares : {0: 94, 1: 12, 8: 4}
Seuil du test : p < 9e-05


,clients rares,mécontents parmi eux,taux,p
colonne,,,,
imp_op_var40_efect_ult1,29,8,27.59 %,1.2e-05
imp_op_var40_efect_ult3,33,8,24.24 %,3.4e-05
num_op_var40_efect_ult1,29,8,27.59 %,1.2e-05
num_op_var40_efect_ult3,33,8,24.24 %,3.4e-05


110 colonnes ont une valeur qui occupe au moins 99,9 % des lignes : au plus 60 clients s'en écartent, et on attend ≈ 2,4 mécontents parmi eux. Pour 106 colonnes, ce qu'on observe (0 ou 1 mécontent) est compatible avec le hasard : c'est du bruit que le modèle apprendrait par cœur.

Mais **4 colonnes** (`*_op_var40_efect_ult1/ult3`) ont **24 à 28 %** de mécontents parmi leurs ~30 clients rares, soit 6 à 7 fois la moyenne. Le test binomial le confirme, même corrigé pour les 110 tests (p ≈ 10⁻⁵).

**Décision : supprimer 106 quasi constantes, garder les 4 qui ont un effet mesuré.** Le signal ne repose que sur 8 mécontents : **essai à journaliser** avec et sans elles.

### Ce qui reste

In [10]:
RESTANTES = [c for c in VARIABLES if c not in CONSTANTES and c not in QUASI_SUPPRIMEES]
print(f"Variables gardées : {len(RESTANTES)} sur {len(VARIABLES)}")
classes = pd.cut(n_tr[RESTANTES], [1, 2, 10, 100, np.inf], labels=["2 (binaire)", "3 à 10", "11 à 100", "> 100"])
display(pd.crosstab(pd.Series([famille(c) for c in RESTANTES], index=RESTANTES, name="famille"),
                    classes.rename("valeurs distinctes")))
print("Variables isolées :", {c: int(n_tr[c]) for c in ["var3", "var15", "var21", "var36", "var38"]})
num = [c for c in VARIABLES if famille(c) == "num"]
print(f"Colonnes num_ qui ne contiennent que des multiples de 3 : {sum(bool((train[c] % 3 == 0).all()) for c in num)} sur {len(num)}")

Variables gardées : 227 sur 369


valeurs distinctes,2 (binaire),3 à 10,11 à 100,> 100
famille,,,,
delta,0,2,2,0
imp,0,0,4,19
ind,53,0,0,0
isolée,0,1,2,2
num,6,50,39,1
saldo,0,0,6,40


Variables isolées : {'var3': 195, 'var15': 99, 'var21': 22, 'var36': 5, 'var38': 46577}
Colonnes num_ qui ne contiennent que des multiples de 3 : 131 sur 143


**Constat** : 227 variables restent. `var15` (âge) et `var38` sont continues ; `var36` (5 valeurs) est une catégorie ; `var3` est à 97,5 % égale à 2. **131 colonnes `num_` sur 143 ne contiennent que des multiples de 3** : les comptages sont codés × 3 (sans effet sur les arbres ni sur l'AUC, mais à savoir avant d'interpréter un comptage).

**Décision : `var36` en catégories, tout le reste en numérique.** Aucune variable à forte cardinalité n'est une catégorie déguisée : pas d'encodage par la cible (et donc pas de risque de fuite). Les colonnes identiques entre elles relèvent de **1.7 Doublons (P4)**.

## 1.9 — Cibles

**Question** : combien de mécontents, à quoi ressemblent-ils, et qu'est-ce que leur petit nombre change à la mesure ?

In [11]:
n, k = len(y), int(y.sum())
p = k / n
marge = 1.96 * np.sqrt(p * (1 - p) / n)
print(f"TARGET : valeurs {sorted(int(v) for v in y.unique())}, manquantes {int(y.isna().sum())}")
print(f"Mécontents : {k} / {n} = {pct(p)}  (intervalle à 95 % : {pct(p - marge)} à {pct(p + marge)})")
print(f"1 mécontent pour {(n - k) / k:.0f} satisfaits · « personne n'est mécontent » = {pct(1 - p)} d'exactitude")
print(f"Attendus dans a_decider au même taux : {round(p * len(a_decider))}")

TARGET : valeurs [0, 1], manquantes 0
Mécontents : 2406 / 60816 = 3.96 %  (intervalle à 95 % : 3.80 % à 4.11 %)
1 mécontent pour 24 satisfaits · « personne n'est mécontent » = 96.04 % d'exactitude
Attendus dans a_decider au même taux : 602


### À quoi ressemble un mécontent

In [12]:
age = profil(train.groupby(pd.cut(train["var15"], [0, 22, 25, 30, 40, 50, 60, 120]), observed=True)[CIBLE], "âge")
age.index = ["22 ans et moins"] + [f"{int(i.left) + 1} à {int(i.right)}" for i in age.index[1:-1]] + ["61 et plus"]
display(age)
montants = [c for c in VARIABLES if famille(c) in ("saldo", "imp")]
sans_montant = (train[montants] == 0).all(axis=1)
print(f"Clients sans aucun montant ({pct(sans_montant.mean())}) : {pct(y[sans_montant].mean())} de mécontents, "
      f"contre {pct(y[~sans_montant].mean())}")

,taux de mécontents,clients
22 ans et moins,0.00 %,963
23 à 25,0.91 %,24536
26 à 30,4.24 %,9349
31 à 40,6.58 %,11746
41 à 50,7.71 %,7539
51 à 60,7.80 %,3757
61 et plus,4.78 %,2926


Clients sans aucun montant (24.83 %) : 9.14 % de mécontents, contre 2.24 %


**Aucun** mécontent jusqu'à 22 ans, moins de 1 % de 23 à 25 ans, puis un taux multiplié par 8 après 40 ans. Les clients aux comptes vides sont 4 fois plus souvent mécontents. **Portrait : un client de plus de 30 ans, avec peu de produits et des comptes vides.**

### La part que personne ne peut prédire

In [13]:
nb_cibles = train.groupby(VARIABLES, sort=False)[CIBLE].transform("nunique")
ambigus = nb_cibles > 1
print(f"{int(ambigus.sum())} clients ont un dossier identique à un client de cible opposée, "
      f"dont {int(y[ambigus].sum())} mécontents ({pct(y[ambigus].sum() / k)} des mécontents)")

1994 clients ont un dossier identique à un client de cible opposée, dont 256 mécontents (10.64 % des mécontents)


À dossier égal, aucun modèle ne peut départager ces clients : c'est du **bruit de cible** irréductible. L'AUC maximale atteignable est donc **inférieure à 1**.

### Ce que le petit nombre de mécontents fait à la mesure

In [14]:
# --- Protocole de validation et règle à une variable -------------------------

# 80 / 20 stratifié sur TARGET, graine fixe : chaque partie garde 3,96 % de mécontents
i_app, i_val = train_test_split(train.index, test_size=PART_VALIDATION, stratify=y, random_state=GRAINE)

# Chaque variable prise seule comme score. Le sens (grand = mécontent, ou l'inverse)
# est choisi sur l'apprentissage ; l'AUC est mesurée sur la validation, jamais vue.
lignes = []
for c in VARIABLES:
    if train.loc[i_app, c].nunique() < 2:
        continue
    sens = 1 if roc_auc_score(y[i_app], train.loc[i_app, c]) >= 0.5 else -1
    lignes.append((c, sens, roc_auc_score(y[i_val], sens * train.loc[i_val, c])))
regles = (pd.DataFrame(lignes, columns=["variable", "sens", "AUC validation"])
          .sort_values("AUC validation", ascending=False).set_index("variable"))

# Hypothèse : la règle de la banque est celle dont l'AUC est la plus proche du plancher
REGLE = (regles["AUC validation"] - PLANCHER).abs().idxmin()
SENS = int(regles.loc[REGLE, "sens"])
y_val = y[i_val].to_numpy()
score_val = SENS * train.loc[i_val, REGLE].to_numpy()
print(f"validation : {len(i_val)} clients, {int(y_val.sum())} mécontents · règle métier : {REGLE} "
      f"(AUC {regles.loc[REGLE, 'AUC validation']:.4f})")

validation : 12164 clients, 481 mécontents · règle métier : var15 (AUC 0.6903)


In [15]:
rng = np.random.default_rng(GRAINE)
tirages = []
for _ in range(300):
    i = rng.integers(0, len(y_val), len(y_val))
    tirages.append(roc_auc_score(y_val[i], score_val[i]))
ecart = np.std(tirages)
print(f"AUC de la règle métier ({REGLE}) : {roc_auc_score(y_val, score_val):.4f} ± {ecart:.4f} (bootstrap, 300 tirages)")
plis = [int(y.iloc[t].sum()) for g in range(20) for _, t in KFold(5, shuffle=True, random_state=g).split(train)]
print(f"Sans stratification, mécontents par pli sur 20 × 5 plis : de {min(plis)} à {max(plis)} (attendu {k / 5:.0f})")

AUC de la règle métier (var15) : 0.6903 ± 0.0099 (bootstrap, 300 tirages)
Sans stratification, mécontents par pli sur 20 × 5 plis : de 433 à 525 (attendu 481)


**Constat** : la validation ne contient que **481 mécontents** ; l'AUC y est mesurée à **± 0,01** près. Un écart de moins de ~0,02 entre deux essais n'est pas distinguable du hasard : battre le plancher de 0,005 ne prouve rien. Sans stratification, un pli peut contenir de 433 à 525 mécontents.

**Décisions**

- **Stratifier tout découpage sur `TARGET`.**
- **Comparer les essais en validation croisée stratifiée** (5 plis, graine 42) pour réduire le bruit de mesure — proposition au mesureur.
- Ne pas s'acharner sur les dossiers ambigus : une partie des mécontents est imprévisible par construction.

## Les décisions de P3, en une page

| # | Constat mesuré | Décision |
|---|---|---|
| 1 | `var3 = -999999` : vrai manquant, 88 clients | remplacer par 2 |
| 2 | `delta_ = 9999999999` : apparition depuis zéro | remplacer par 9 ; pas d'imputation |
| 3 | `var38` : valeur de remplissage chez 20 % des clients, sans effet | garder ; essai « manquant + indicateur » |
| 4 | `var36 = 99` : 40 % des clients, taux le plus haut | encoder en catégories |
| 5 | les zéros sont des mesures | garder ; essai « nombre de valeurs non nulles » |
| 6 | 36 constantes (et non 97) | supprimer |
| 7 | 110 quasi constantes, dont 4 avec un effet réel | supprimer 106, garder 4 ; essai avec / sans |
| 8 | 481 mécontents en validation, AUC à ± 0,01 | stratifier ; validation croisée |
| 9 | 1 994 dossiers identiques de cibles opposées | bruit irréductible, AUC max < 1 |